# SEVIRI: regole RDT-CW-inspired sul dominio completo

Rilevamento, proxy UTLS e tracking vengono calcolati su tutto il dominio. Le clip sono ritagli successivi degli stessi risultati, con ID persistenti lungo la scansione. La scansione corrente copre tutto luglio 2020 (1–31 luglio), con checkpoint incrementali.

La proxy è una temperatura radiometrica dell’incudine, non una retrieval della tropopausa WMO. Questi candidati non sono predizioni VideoMAE e non entrano nel pretraining.


In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

ROOT = Path.cwd()
if not (ROOT/'src').exists(): ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
# Reload the reader too: the existing kernel may still expect directory Zarr.
import emma_gpm.rdt.storage as domain_storage
importlib.reload(domain_storage)
import emma_gpm.rdt_event_explorer as event_explorer
importlib.reload(event_explorer)
import emma_gpm.rdt_teacher as teacher
importlib.reload(teacher)
import emma_gpm.rdt_refined as refined_teacher
importlib.reload(refined_teacher)
import emma_gpm.rdt_context as context_teacher
importlib.reload(context_teacher)
import emma_gpm.rdt_preview as preview
importlib.reload(preview)
import emma_gpm.rdt_dashboard as dashboard
importlib.reload(dashboard)
ZARR_ROOT=Path(os.environ.get('SEVIRI_ZARR_ROOT','/media/fenrir/disk1/Datasets/MSG-SEVIRI'))
DATASET_DIR=ROOT/'results/rdt_preview/domain_utls_2020_parallel'
CONFIG=teacher.RDTConfig()


## Avanzamento del calcolo sul dominio completo

Il contatore misura **frame dell’intero dominio**. Il totale annuale include già luglio; la precedente scansione di luglio è conservata separatamente. L’inventario resta fissato all’avvio.

Il catalogo automatico su griglia è **disattivato**. Il teacher calcola rilevamento, proxy, tracking e pseudo-classi sul dominio completo; la ricerca dei picchi e gli zoom su richiesta sono nel viewer in fondo. I cataloghi precedenti sono conservati come diagnostiche storiche, senza nuovi crop automatici.


In [ ]:
refresh=widgets.Button(description='Aggiorna risultati',icon='refresh')
summary_output=widgets.Output()
viewer_output=widgets.Output()

def reload_results(button=None):
    global viewer,catalog,scan_status
    with summary_output:
        clear_output(wait=True)
        progress=DATASET_DIR/'progress.json'
        scan_status=json.loads(progress.read_text()) if progress.exists() else {'status':'non avviato'}
        unit='frame del dominio' if DATASET_DIR.name.startswith('domain_utls') else 'clip'
        print(f"Scansione: {scan_status.get('status')} · {scan_status.get('processed',0):,}/{scan_status.get('requested',0):,} {unit}")
        plan_path=DATASET_DIR/'parallel_plan.json'
        if plan_path.exists():
            plan=json.loads(plan_path.read_text())
            july_start=pd.Timestamp(f"{plan['year']}-07-01")
            july_end=pd.Timestamp(f"{plan['year']}-08-01")
            july_frames=sum(max(0,int((min(pd.Timestamp(part['end_exclusive']),july_end)-max(pd.Timestamp(part['start']),july_start))/pd.Timedelta(minutes=5))) for part in plan['sequences'])
            print(f"Totale: {plan['frames']:,} frame unici, inclusi {july_frames:,} di luglio già analizzati nella prova precedente; non si sommano.")
            print(f"Snapshot: {pd.Timestamp(plan['first']):%d/%m/%Y}–{pd.Timestamp(plan['last']):%d/%m/%Y}, {plan['days']} giorni con dati. I nuovi download non entrano automaticamente in questo job.")
        if 'active_workers' in scan_status:
            remaining=scan_status['sequences']-scan_status['completed_sequences']
            print(f"Worker di tracking: {scan_status['active_workers']} · sequenze incomplete: {remaining} · limite processi: {scan_status['workers']} · sequenze complete: {scan_status['completed_sequences']}/{scan_status['sequences']}")
            if scan_status.get('detector_workers_per_sequence'):
                print(f"Rilevamento in parallelo: {scan_status.get('active_detection_workers',0)} processi aggiuntivi · salvataggio asincrono: {'sì' if scan_status.get('async_save') else 'no'}.")
            sequence_status_path=DATASET_DIR/'sequence_status.json'
            if sequence_status_path.exists():
                verifying=sum(part.get('status')=='verifying_checkpoint' for part in json.loads(sequence_status_path.read_text()))
                if verifying:
                    print(f"Verifica checksum degli output precedenti: {verifying} sequenze. Il calcolo riprende automaticamente dopo il controllo.")
            print(f"Velocità media storica dall’avvio: {scan_status.get('frames_per_second',0):.2f} frame/s (include le fasi con più processi).")
            current=(str(DATASET_DIR.resolve()),scan_status.get('elapsed_seconds',0),scan_status.get('processed',0))
            previous=globals().get('_rdt_rate_sample')
            if previous and previous[0]==current[0] and current[1]>previous[1] and current[2]>=previous[2]:
                interval=current[1]-previous[1]
                if interval>=30:
                    print(f"Velocità tra gli ultimi aggiornamenti: {(current[2]-previous[2])/interval:.2f} frame/s su {interval:.0f} s.")
                    globals()['_rdt_rate_sample']=current
                else:
                    print('Velocità recente: premi Aggiorna risultati dopo almeno 30 secondi dalla prima lettura.')
            else:
                globals()['_rdt_rate_sample']=current
                print('Velocità recente: premi Aggiorna risultati dopo almeno 30 secondi per misurarla.')
            print('Le sequenze avanzano in parallelo: la data più recente non implica copertura completa dei giorni precedenti.')
        if scan_status.get('error'):
            print(f"Dettaglio errore: {scan_status['error']}")
        if scan_status.get('invalid_radiometry_pixels', 0):
            print(f"Campioni pixel/tempo radiometrici esclusi: {scan_status['invalid_radiometry_pixels']:,}")
        if scan_status.get('crop_catalog_enabled') is False:
            print('Solo dominio completo: catalogo automatico dei crop disattivato. Usa i picchi e Zoom di studio nel viewer in fondo.')
            if 'viewer' in globals():
                viewer.close();del viewer
            with viewer_output:
                clear_output(wait=True)
            return
        path=DATASET_DIR/'catalog.csv'
        if not path.exists():
            print('In attesa della prima finestra completa di 16 frame.');return
        catalog=pd.read_csv(path)
        good=catalog[catalog.status=='ok']
        if good.empty: print('Nessuna clip valida nel blocco corrente.');return
        n_conv=int((good.convective_pixels>0).sum());n_ot=int((good.ot_pixels>0).sum())
        print(f"Clip (16 frame × crop 224×224): valide {len(good):,} · rifiutate: {len(catalog)-len(good):,} · con candidati convettivi: {n_conv:,} · con OT candidati: {n_ot:,}")
        if 'known_cold_pixels' in good and good.cold_pixels.sum()>0:
            print(f"Nube fredda con storia valutabile: {100*good.known_cold_pixels.sum()/good.cold_pixels.sum():.1f}% dei pixel freddi")
        if 'convective_peak_fraction' in good:
            fig,axes=plt.subplots(1,2,figsize=(12,3.5))
            rows=[]
            for month,group in good.groupby('month'):
                rows.append((int(month),100*(group.convective_pixels>0).mean(),100*(group.ot_pixels>0).mean()))
            stats=np.asarray(rows);x=np.arange(len(rows))
            axes[0].bar(x-.18,stats[:,1],width=.36,label='Convezione candidata')
            axes[0].bar(x+.18,stats[:,2],width=.36,label='OT candidato')
            axes[0].set_xticks(x,[f'{int(month):02d}' for month in stats[:,0]])
            axes[0].set(xlabel='Mese',ylabel='% delle clip valide',title='Presenza di candidati');axes[0].legend()
            positive=good[good.convective_pixels>0]
            axes[1].hist(100*positive.convective_peak_fraction,bins=20)
            axes[1].set(xlabel='% dei pixel nel frame più attivo',ylabel='Clip',title='Occupazione: solo clip con candidati')
            fig.tight_layout();display(fig);plt.close(fig)
    if 'viewer' in globals():viewer.close()
    viewer=dashboard.RDTClipViewer(ZARR_ROOT,catalog,CONFIG,warmup_frames=3)
    with viewer_output:
        clear_output(wait=True);display(viewer.widget)

refresh.on_click(reload_results)
display(refresh,summary_output)
reload_results()


## Studio dei fenomeni

Usa il viewer del **dominio completo**, più sotto, per cercare i picchi e aprire gli **Zoom di studio**. Non vengono più generati esempi automatici su griglia. Gli zoom consentono l’ispezione del teacher e la selezione dei casi; il dataset di probing sarà definito separatamente con split temporali e copertura delle cinque classi.

Gli ID annuali identificano sequenze e tracce: `blocco × 2**32 + contatore locale`. Non contano gli eventi; 0 indica lo sfondo. Una traccia mantiene il proprio ID nelle clip in cui compare.


In [ ]:
display(viewer_output)

## Dati e ripresa

La scansione annuale procede fuori dal notebook. **Aggiorna risultati** rilegge le clip già pubblicate; nel player del dominio completo scegli archivio e giorno, poi usa **Aggiorna** per vedere i nuovi frame.

Stato in `results/rdt_preview/domain_utls_2020_parallel/progress.json`; dettaglio delle 20 sequenze in `sequence_status.json`; log generale `scan.log` e log dei processi in `segments/NNN/training.log`. Rilanciare lo stesso comando riprende dai checkpoint locali:

```bash
.venv/bin/python scripts/build_rdt_year_parallel.py --year 2020 --workers 20
```

Copertura disponibile nello snapshot: 317 giorni, dal 1 gennaio al 9 dicembre 2020, con gap espliciti. Nessun frame mancante viene sostituito. Le sequenze sono separate solo da gap reali: il tracking prosegue attraverso la mezzanotte. Gli ID annuali sono globalmente univoci, quindi numericamente grandi; non sono conteggi di eventi.


## Dominio completo: picchi annuali, giornate e zoom

**Ricerca** sceglie tra il giorno visualizzato e tutte le giornate già calcolate dell’archivio selezionato. I pulsanti **Picco convettivo** e **Picco OT** cercano la massima percentuale di pixel validi occupati dalle rispettive classi; il menu **Picchi** mostra altri massimi, distanziati di almeno 90 minuti per evitare frame quasi identici. Selezionare un picco sposta automaticamente giorno e frame. **Rileggi giornate** aggiorna risultati e graduatorie.

L’indice viene calcolato separatamente e può essere in ritardo rispetto al teacher: il numero di frame indicizzati è sempre visibile. Durante la scansione, le graduatorie riguardano soltanto i risultati già indicizzati, non tutto l’anno ancora da elaborare.

In **Zoom di studio** scegli una traccia tra le celle più estese (km²) o con più pixel OT persistenti. **Zoom · clip 16 frame** centra un crop 224×224 sulla traccia al suo picco, con una finestra temporale completa il più possibile centrata sul picco. Il crop rimane fisso nello spazio e può contenere soltanto parte di una nube molto estesa. Nessun gap viene riempito. Se mancano frame o il crop contiene dati invalidi, lo zoom viene rifiutato.

La selezione di studio richiede supporto della proxy e assenza di contatto col bordo; per le celle convettive richiede anche storia e fase valutabili. Una traccia non equivale sempre a un evento meteorologico indipendente, specialmente dopo split/merge. Le graduatorie misurano estensione e occupazione OT, non intensità meteorologica validata. Sono diagnostiche separate dal campionamento SSL.


In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, importlib
from IPython.display import display

DOMAIN_ROOT = Path.cwd()
if not (DOMAIN_ROOT/'src').exists(): DOMAIN_ROOT = DOMAIN_ROOT.parent
sys.path.insert(0, str(DOMAIN_ROOT/'src'))
# Reload the reader too: the existing kernel may still expect directory Zarr.
import emma_gpm.rdt.storage as domain_storage
importlib.reload(domain_storage)
import emma_gpm.rdt_event_explorer as event_explorer
importlib.reload(event_explorer)
import emma_gpm.rdt_dashboard as clip_dashboard
importlib.reload(clip_dashboard)
import emma_gpm.rdt_domain_dashboard as domain_dashboard
importlib.reload(domain_dashboard)
DOMAIN_ZARR_ROOT = Path(os.environ.get('SEVIRI_ZARR_ROOT', '/media/fenrir/disk1/Datasets/MSG-SEVIRI'))
ANNUAL_ARCHIVE = DOMAIN_ROOT/'results/rdt_preview/domain_utls_2020_parallel'
DOMAIN_ARCHIVES = {
    **({'2020 · scansione parallela': ANNUAL_ARCHIVE} if (ANNUAL_ARCHIVE/'protocol.json').exists() else {}),
    '11 luglio · prova completa': DOMAIN_ROOT/'results/rdt_preview/domain_utls_july2020',
    'Luglio · scansione mensile': DOMAIN_ROOT/'results/rdt_preview/domain_utls_july2020_month',
}
if 'domain_day_viewer' in globals(): domain_day_viewer.close()
domain_day_viewer = domain_dashboard.RDTDomainDayViewer(
    DOMAIN_ZARR_ROOT, DOMAIN_ARCHIVES, preferred_day='2020-07-11')


In [ ]:
display(domain_day_viewer.widget)


### Attività lungo la giornata

Percentuale **per frame** dei pixel validi classificati come convezione candidata e OT persistenti, nella stessa unità della graduatoria. Frame mancanti o non ancora indicizzati rimangono interruzioni nelle curve. Se l’indice del giorno non è pronto, vengono mostrati temporaneamente i conteggi già disponibili, con unità esplicite.

I pulsanti **Picco convettivo** e **Picco OT** seguono il perimetro scelto in **Ricerca**: giorno corrente oppure tutte le giornate calcolate dell’archivio selezionato.


In [ ]:
display(domain_day_viewer.activity_output)


## Differenza WV6.2 − IR10.8: scala e campione statistico

Le mappe usano una scala **fissa −80/+10 K**, con bianco a **0 K**, blu per le differenze negative e rosso per quelle positive. La trasformazione del colore è lineare separatamente sui due lati dello zero: lo stesso incremento di K può cambiare colore più rapidamente sul lato positivo. La colorbar segnala eventuali valori oltre i limiti. Questa scala non è una stima degli estremi annuali e non modifica dati o regole.

Il riepilogo seguente usa un campione con seed fisso: tre frame del dominio e una clip casuale completa per ogni mese disponibile, con rifiuto esplicito delle clip non valide. I percentili e le frequenze si riferiscono al campione; non sono statistiche esaustive dell’intero archivio. I crop del catalogo RDT principale sono invece su griglia regolare, ogni 80 minuti, senza random sampling.


In [ ]:
from pathlib import Path
import json
import pandas as pd
BTD_ROOT=Path.cwd()
if not (BTD_ROOT/'src').exists(): BTD_ROOT=BTD_ROOT.parent
statistics_path=BTD_ROOT/'results/rdt_preview/domain_utls_2020_parallel/exploration/btd_statistics.json'
if statistics_path.exists():
    btd_report=json.loads(statistics_path.read_text())
    rows=[]
    for title,key in [('Dominio · 36 frame campionati','domain'),('Crop · 12 clip complete','clips')]:
        values=btd_report[key]
        if values:
            rows.append({'Campione':title,'Mediana (K)':round(values['median_k'],2),
                         'P1–P99 (K)':f"{values['p01_k']:.1f} … {values['p99_k']:.1f}",
                         'Pixel positivi (%)':round(values['positive_percent'],3)})
    display(pd.DataFrame(rows).set_index('Campione'))
else:
    print('Statistiche BTD ancora in preparazione.')
